# serve

> MCP server example for exposing yttoc tools to Claude Code, Codex, and other MCP clients

In [ ]:
#| default_exp serve

## How MCP works

MCP (Model Context Protocol) lets LLMs call external tools over JSON-RPC on stdio.

The flow:

```
LLM client (Claude Code, Codex, etc.)
  <-> stdin/stdout (JSON-RPC)
MCP server (this module)
  <-> calls Python functions / shell commands
Local tools (date, yttoc, etc.)
```

Three concepts:
1. **Server**: `FastMCP("name")` creates an MCP server instance
2. **Tool**: `mcp.tool()(function)` registers a Python function as a callable tool
3. **Run**: `mcp.run()` starts the stdio transport and blocks

The LLM sees each registered function as a tool it can call, just like built-in tools.

## Step 1: Define the callable work

Keep the real work in a plain function first. This makes it testable without starting the MCP transport and avoids importing optional MCP dependencies at module import time.

In [ ]:
#| export
import subprocess

In [ ]:
#| export
def get_date(utc: bool = False) -> str:
    "Return the current date and time from the system clock."
    cmd = ["date", "--utc"] if utc else ["date"]
    return subprocess.run(cmd, capture_output=True, text=True, check=True).stdout.strip()

## Step 2: Build the MCP server

`mcp` is an optional dependency, so import it inside `build_server()`. A normal `import yttoc.serve` should work even when the MCP extra is not installed.

In [ ]:
#| export
def build_server():
    "Create the MCP server and register yttoc tools."
    try:
        from mcp.server.fastmcp import FastMCP
    except ModuleNotFoundError as e:
        raise RuntimeError('Install MCP support with: pip install -e ".[mcp]"') from e

    mcp = FastMCP("yttoc")
    mcp.tool()(get_date)
    return mcp

## Step 3: Entry point

`mcp.run()` starts the stdio JSON-RPC transport. This is what `claude mcp add` and `codex mcp add` invoke.

In [ ]:
#| export
def main():
    "Entry point for yttoc-serve CLI command."
    build_server().run()

## Test

We can verify the tool function directly without MCP transport:

In [ ]:
result = get_date()
print(result)
assert len(result) > 0

result_utc = get_date(utc=True)
print(result_utc)
assert "UTC" in result_utc
print("ok")

In [ ]:
#| eval: false
server = build_server()
assert server.name == "yttoc"

## How to register

After `nbdev-export` and `pip install -e ".[mcp]"`, register with each LLM client from the repository root:

```bash
# Claude Code
claude mcp add yttoc -- "$(pwd)/.venv/bin/yttoc-serve"

# Codex
codex mcp add yttoc -- "$(pwd)/.venv/bin/yttoc-serve"
```

Then restart the LLM client. The tool appears as `mcp__yttoc__get_date` and the LLM can call it directly.